In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/home/ec2-user/SageMaker/amazon-ml-challenge")

sys.path.append(str(PROJECT_ROOT / "src"))

from utils import load_ground_truth

In [2]:
import pandas as pd

DATA_DIR = PROJECT_ROOT / "data" / "raw"

def load_ids(path):
    ids = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id"],
        dtype=str,
        chunksize=200_000,
    ):
        ids.update(chunk["entity_id"])

    return ids


print("Loading S2 IDs...")
train_s2_ids = load_ids(DATA_DIR / "train_source2.tsv")

print("Loading S3 IDs...")
train_s3_ids = load_ids(DATA_DIR / "train_source3.tsv")

print("S2 IDs:", f"{len(train_s2_ids):,}")
print("S3 IDs:", f"{len(train_s3_ids):,}")

Loading S2 IDs...
Loading S3 IDs...
S2 IDs: 1,726,856
S3 IDs: 1,892,297


In [3]:
print("Loading ground truth...")

safe_gt, unsafe_ids = load_ground_truth(
    DATA_DIR / "train_ground_truth.tsv",
    train_s2_ids,
    train_s3_ids,
)

print("\nRESULT")
print("=" * 60)

print("Safe S1 entities:", f"{len(safe_gt):,}")
print("Unsafe S1 entities:", f"{len(unsafe_ids):,}")

safe_singletons = sum(
    1 for matches in safe_gt.values()
    if len(matches) == 0
)

safe_matched = len(safe_gt) - safe_singletons

print("Safe singletons:", f"{safe_singletons:,}")
print("Safe matched entities:", f"{safe_matched:,}")

Loading ground truth...

RESULT
Safe S1 entities: 243,314
Unsafe S1 entities: 1,963,507
Safe singletons: 123,247
Safe matched entities: 120,067


In [4]:
from collections import Counter

print("=" * 70)
print("CHECKING MATCH-ID REUSE")
print("=" * 70)

match_to_s1 = Counter()
total_links = 0

for s1_id, matches in safe_gt.items():
    for entity_id in matches:
        match_to_s1[entity_id] += 1
        total_links += 1

print("Total usable positive links:", f"{total_links:,}")
print("Unique matched S2/S3 IDs:", f"{len(match_to_s1):,}")

reused = {
    entity_id: count
    for entity_id, count in match_to_s1.items()
    if count > 1
}

print("S2/S3 IDs matched to >1 S1:", f"{len(reused):,}")

if reused:
    print("\nMost reused IDs:")
    for entity_id, count in sorted(
        reused.items(),
        key=lambda x: x[1],
        reverse=True
    )[:20]:
        print(f"  {entity_id}: {count}")
else:
    print("\nNo match IDs are reused across safe S1 entities.")

CHECKING MATCH-ID REUSE
Total usable positive links: 242,487
Unique matched S2/S3 IDs: 242,487
S2/S3 IDs matched to >1 S1: 0

No match IDs are reused across safe S1 entities.


In [5]:
import random

# Reproducible split
random.seed(42)

safe_s1_ids = list(safe_gt.keys())

random.shuffle(safe_s1_ids)

split_idx = int(len(safe_s1_ids) * 0.80)

train_s1_ids = safe_s1_ids[:split_idx]
valid_s1_ids = safe_s1_ids[split_idx:]

train_gt = {
    s1_id: safe_gt[s1_id]
    for s1_id in train_s1_ids
}

valid_gt = {
    s1_id: safe_gt[s1_id]
    for s1_id in valid_s1_ids
}

print("TRAIN / VALIDATION SPLIT")
print("=" * 60)

print("Total safe S1:", f"{len(safe_s1_ids):,}")
print("Training S1:", f"{len(train_s1_ids):,}")
print("Validation S1:", f"{len(valid_s1_ids):,}")

print("\nTraining singletons:",
      f"{sum(len(v) == 0 for v in train_gt.values()):,}")

print("Validation singletons:",
      f"{sum(len(v) == 0 for v in valid_gt.values()):,}")

print("\nTraining positive links:",
      f"{sum(len(v) for v in train_gt.values()):,}")

print("Validation positive links:",
      f"{sum(len(v) for v in valid_gt.values()):,}")

TRAIN / VALIDATION SPLIT
Total safe S1: 243,314
Training S1: 194,651
Validation S1: 48,663

Training singletons: 98,379
Validation singletons: 24,868

Training positive links: 194,566
Validation positive links: 47,921


In [6]:
import json
from pathlib import Path

SPLIT_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

with open(SPLIT_DIR / "train_s1_ids.json", "w") as f:
    json.dump(train_s1_ids, f)

with open(SPLIT_DIR / "valid_s1_ids.json", "w") as f:
    json.dump(valid_s1_ids, f)

print("Saved validation split.")

Saved validation split.
